# SaShiMi + DiffWave — music continuation (Colab quickstart)
Unconditional raw-waveform generation at 22.05 kHz / ~2 s clips on a single A100 or L4 (Colab Pro).

**Just drop ONE long audio file on Drive** (any format/length — a 4-hour mp3 is fine). Step 4 decodes → mono → resamples → splits it automatically with **bounded memory** (ffmpeg streaming), then training chunks it on the fly.

**Runtime → Change runtime type → GPU (A100 or L4).** Do *not* pip-install torch/torchaudio; use Colab's preinstalled CUDA-matched build.

In [ ]:
# 1. Clone the project from GitHub, and mount Drive for persistent checkpoints.
!git clone https://github.com/heyuwang1999/diffwave-sashimi.git
%cd diffwave-sashimi
from google.colab import drive; drive.mount('/content/drive')
# Persist checkpoints/samples to Drive (repo's vendor/exp is gitignored; /content is wiped on disconnect).
!mkdir -p /content/drive/MyDrive/diffwave-sashimi-exp && ln -sfn /content/drive/MyDrive/diffwave-sashimi-exp vendor/exp

In [ ]:
# 2. Confirm GPU + install Python deps (NOT torch/torchaudio). ffmpeg is preinstalled on Colab.
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip install -q -r requirements-colab.txt

In [ ]:
# 3. (optional) Sanity-check the whole stack on the real GPU before training (~30s).
%cd vendor
!python ../scripts/smoke_test.py
%cd ..

In [ ]:
# 4. AUTOMATIC PROCESSING: point at your file (or a folder) on Drive and run.
#    Works on a single multi-hour file; ffmpeg streams it so memory stays bounded.
#    Output: vendor/data/music (train) + vendor/data/music_holdout (last 10%, for continuation eval).
AUDIO_SRC = '/content/drive/MyDrive/mymix.mp3'   # <-- your long file (any format) OR a folder of files
!python scripts/prepare_data.py --in_dir "{AUDIO_SRC}" --out_dir vendor/data/music --sr 22050 --holdout_frac 0.1

In [ ]:
# 5. Train (run from vendor/ — Hydra config_path and imports are anchored there).
#    batch_size_per_gpu: 4 fits a 22-24GB L4/A10 at d_model=64, L~44k.
#    OOM? -> lower to 2 or 1.  A100 40GB -> 8 (or 16).  expandable_segments reduces fragmentation.
%cd vendor
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py experiment=music train.batch_size_per_gpu=4
# Bump quality once it trains: model.d_model=128 (needs more memory -> smaller batch)
# Fidelity stack (A/B vs baseline): diffusion.parameterization=v diffusion.min_snr_gamma=5.0 \
#   diffusion.schedule=cosine diffusion.stft_loss_weight=0.1 model.self_conditioning=true train.name=fidelity_stack
# Continuation: python train.py experiment=music_continuation   (or experiment=music_continuation_xattn)

In [ ]:
# 6. Generate from the latest checkpoint.
!python generate.py experiment=music generate.ckpt_iter=max generate.n_samples=8
# Faster: generate.sampler=ddim generate.sampling_steps=20
# Continuation, CUSTOM LENGTH (e.g. 60s): python generate.py experiment=music_continuation \
#   generate.ckpt_iter=max generate.context_path=data/music_holdout/<track>.wav \
#   generate.guidance=3.0 generate.gen_seconds=60
# WAVs land in vendor/exp/<run>/waveforms/  (continuation also writes *_full.wav = context+gen)

In [ ]:
# 7. Listen to a generated sample.
import glob, IPython.display as ipd
wavs = sorted(glob.glob('exp/**/waveforms/**/*.wav', recursive=True))
print(wavs[-3:])
ipd.Audio(wavs[-1]) if wavs else print('No samples found yet.')

In [ ]:
# 8. Evaluate (A/B): Frechet Mel Distance of generated vs real reference set.
!python ../scripts/evaluate.py fidelity --gen_dir exp/<run>/waveforms/<iter> --ref_dir data/music --sr 22050